In [1]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER
import os

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'BBVA', '2206.pdf')
statement_path

'c:\\Proyectos\\Aether\\Aether_v1\\documents\\inputs\\test_files\\BBVA\\2206.pdf'

In [2]:
from document_reader import PDFReader

reader = PDFReader(statement_path)
extracted_words = reader.extract_words_from_pdf()

extracted_words.head()

,page,page_height,text,x0,top,x1,bottom
0,1,792,Estado,510.546000,1.21900,544.314000,13.21900
1,1,792,de,546.858000,1.21900,559.638000,13.21900
2,1,792,Cuenta,562.182000,1.21900,598.182000,13.21900
3,1,792,Libretón,459.986607,16.65441,498.904607,27.65441
4,1,792,Básico,501.236607,16.65441,530.397608,27.65441


In [3]:
from bank_detector import DefaultBankDetector

bank_detector = DefaultBankDetector(extracted_words)

bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.statement_propertys

print(f'{bank} - {statement_type}')
statement_properties

bbva - debit


{'start_phrase': ['detalle', 'de', 'movimientos', 'realizados'],
 'end_phrase': ['le', 'informamos', 'que', 'puede'],
 'columns': ['OPER',
  'LIQ',
  'DESCRIPCION',
  'REFERENCIA',
  'CARGOS',
  'ABONOS',
  'OPERACION',
  'LIQUIDACION'],
 'date_column': 'OPER',
 'description_column': 'DESCRIPCION',
 'amount_column': ['CARGOS', 'ABONOS'],
 'income_column': 'ABONOS',
 'expense_column': 'CARGOS',
 'date_pattern': '^(\\d{2})/([A-Z]{3})\\b',
 'date_groups': (None, 2, 1),
 'month_pattern': {'ENE': '01',
  'FEB': '02',
  'MAR': '03',
  'ABR': '04',
  'MAY': '05',
  'JUN': '06',
  'JUL': '07',
  'AGO': '08',
  'SEP': '09',
  'OCT': '10',
  'NOV': '11',
  'DIC': '12'},
 'period_phrase': ['periodo'],
 'period_pattern': '(\\d{2})/(\\d{2})/(\\d{4})',
 'year_group': 3}

In [4]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

df_table = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
df_table.head()

187 - 886


,page,page_height,text,x0,top,x1,bottom
0,1,792,FECHA,39.098979,516.402066,67.328979,526.402066
1,1,792,SALDO,518.600629,516.402066,547.920629,526.402066
2,1,792,OPER,21.333979,526.402066,43.843979,536.402066
3,1,792,LIQ,70.213979,526.402066,84.213979,536.402066
4,1,792,DESCRIPCION,104.588979,526.402066,161.648979,536.402066


In [5]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(df_table)

grouped_rows = row_segmenter.group_rows()
grouped_rows.head()

,row_group,text,x0,x1,top,bottom,page
0,0,FECHA SALDO,"[39.098978999999886, 518.6006289999999]","[67.32897899999989, 547.9206289999998]",516.402066,526.402066,1
1,1,OPER LIQ DESCRIPCION REFERENCIA CARGOS ABONOS ...,"[21.333978999999886, 70.21397899999988, 104.58...","[43.84397899999989, 84.21397899999988, 161.648...",526.402066,536.402066,1
2,2,15/MAY 16/MAY PAGO CUENTA DE TERCERO 40.00,"[14.803975999999807, 59.42897599999981, 106.83...","[50.373975999999814, 94.99897599999981, 134.29...",543.710649,553.710649,1
3,3,Referencia 1634723960 BNET 1592044105 PAGO PRE...,"[318.5299959999998, 367.10349599999984, 106.17...","[364.4624959999998, 419.92349599999983, 130.18...",555.469940,564.969940,1
4,4,15/MAY 16/MAY SPEI ENVIADO SANTANDER 25.00,"[14.803975999999864, 59.428975999999864, 106.8...","[50.37397599999987, 94.99897599999987, 128.489...",566.229231,576.229231,1


In [6]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, statement_properties)

df_table = table_reconstructor.get_structured_table()

df_table

,OPER,LIQ,DESCRIPCION,REFERENCIA,CARGOS,ABONOS,OPERACION,LIQUIDACION
0,15/MAY,,PAGO CUENTA BNET 1592044105,,40.00,,,
1,15/MAY,,SPEI ENVIADO 1505220PAGO 00014910605957017245 ...,,25.00,,,
2,15/MAY,,PAGO CUENTA BNET 1592044105,,60.00,,"266,326.99","266,451.99"
3,17/MAY,,SPEI ENVIADO 1705220PAGO 00005579070112406429 ...,,850.00,,"265,476.99","265,476.99"
4,18/MAY,,SPEI ENVIADO 1805220INVERSION 0064618011541057...,,"80,000.00",,,
5,18/MAY,,SPEI ENVIADO 1805220PAGO 00646180115410578304 ...,,500.00,,,
6,18/MAY,,SAT REF:042229R3450034953290,,"13,621.00",,,
7,18/MAY,,SAT REF:042229QU050035295243,,"7,398.00",,,
8,18/MAY,,SAT REF:042229QD920035292297,,"3,832.00",,,
9,18/MAY,,SAT REF:042229Q9950035295203,,"4,971.00",,,


In [7]:
from table_normalizer import TransactionTableNormalizer

normalizer = TransactionTableNormalizer(df_table, extracted_words, statement_properties)

normalizer.normalize_table() # Final result

,Date,Description,Amount,Type
0,2022-05-15,PAGO CUENTA BNET 1592044105,-40.00,Cargo
1,2022-05-15,SPEI ENVIADO 1505220PAGO 00014910605957017245 ...,-25.00,Cargo
2,2022-05-15,PAGO CUENTA BNET 1592044105,-60.00,Cargo
3,2022-05-17,SPEI ENVIADO 1705220PAGO 00005579070112406429 ...,-850.00,Cargo
4,2022-05-18,SPEI ENVIADO 1805220INVERSION 0064618011541057...,-80000.00,Cargo
5,2022-05-18,SPEI ENVIADO 1805220PAGO 00646180115410578304 ...,-500.00,Cargo
6,2022-05-18,SAT REF:042229R3450034953290,-13621.00,Cargo
7,2022-05-18,SAT REF:042229QU050035295243,-7398.00,Cargo
8,2022-05-18,SAT REF:042229QD920035292297,-3832.00,Cargo
9,2022-05-18,SAT REF:042229Q9950035295203,-4971.00,Cargo
